In [1]:
from pathlib import Path
import re, json
from collections import Counter

TRACE_DIR = Path("traces_clean")
OUT_JSONL = Path("scounteren_path_summaries.jsonl")
OUT_TXT = Path("scounteren_path_summaries.txt")

def split_trace_blocks(text):
    lines = text.splitlines()
    starts = [i for i, line in enumerate(lines) if line.strip().startswith("(trace")]
    if not starts:
        return [text] if text.strip() else []
    blocks = []
    for i, start in enumerate(starts):
        end = starts[i + 1] if i + 1 < len(starts) else len(lines)
        block = "\n".join(lines[start:end]).strip()
        if block:
            blocks.append(block)
    return blocks

def tool_error(text):
    low = text.lower()
    return any(x in low for x in ["panic", "panicked", "dumped core", "timeout", "error:"])

def illegal_path(text):
    return "write-reg |mcause| nil (_ struct (|bits| #x0000000000000002))" in text

def success_path(text):
    return "write-reg |x5|" in text and "RETIRE_SUCCESS" in text and not illegal_path(text)

def priv_from_name(name):
    m = re.search(r"__(Machine|Supervisor|User)__", name)
    return m.group(1) if m else "Unknown"

def experiment_from_name(name):
    if name.startswith("csr_read__"):
        return "csr_read_access"
    if name.startswith("write_probe__"):
        return "write_legalize_probe"
    if name.startswith("counter_access__"):
        return "counter_access_effect"
    return "unknown"

def instr_from_name(name):
    if name.startswith("csr_read__"):
        return "csrr x5, scounteren"
    if name.startswith("write_probe__"):
        return "csrw scounteren, x6"
    if name.startswith("counter_access__"):
        m = re.search(r"counter_access__(Machine|Supervisor|User)__(.+)\.trace", name)
        if m:
            return f"csrr x5, {m.group(2)}"
    return "unknown"

def counter_from_name(name):
    m = re.search(r"counter_access__(Machine|Supervisor|User)__(.+)\.trace", name)
    return m.group(2) if m else None

def written_value_from_name(name):
    m = re.search(r"value_([0-9a-fA-F]{16})", name)
    return m.group(1).lower() if m else None

def important_events(block):
    keys = [
        "assume-reg |cur_privilege|",
        "read-reg |cur_privilege|",
        "read-reg |misa|",
        "read-reg |scounteren|",
        "read-reg |mcounteren|",
        "read-reg |mcycle|",
        "read-reg |mtime|",
        "read-reg |minstret|",
        "read-reg |mhpmcounter",
        "write-reg |x5|",
        "write-reg |mcause|",
        "write-reg |cur_privilege|",
        "Retired",
        "error:",
        "panic",
        "panicked",
    ]
    out = []
    for line in block.splitlines():
        s = line.strip()
        if any(k in s for k in keys):
            out.append(s)
    return out[:140]

rows = []

for path in sorted(TRACE_DIR.glob("*.trace")):
    text = path.read_text(errors="replace")
    blocks = split_trace_blocks(text)

    exp = experiment_from_name(path.name)
    priv = priv_from_name(path.name)
    instr = instr_from_name(path.name)
    counter = counter_from_name(path.name)
    written = written_value_from_name(path.name)

    for i, block in enumerate(blocks):
        terr = tool_error(block)
        illegal = illegal_path(block)
        success = success_path(block)

        if terr:
            status = "tool_error"
            exception = "tool_error"
        elif illegal:
            status = "trapped"
            exception = "illegal_instruction"
        elif success:
            status = "success"
            exception = "none"
        else:
            status = "unknown"
            exception = "unknown"

        if exp == "csr_read_access":
            if status == "success":
                path_name = f"{priv}-mode scounteren read succeeds"
                summary = f"{priv}-mode reads scounteren successfully and writes the value to x5."
            elif status == "trapped":
                path_name = f"{priv}-mode scounteren read traps"
                summary = f"{priv}-mode tries to read scounteren and raises illegal instruction."
            else:
                path_name = f"{priv}-mode scounteren read {status}"
                summary = f"{priv}-mode scounteren read produced status {status}."
        elif exp == "write_legalize_probe":
            path_name = f"{priv}-mode scounteren write/legalize probe"
            if status == "tool_error":
                summary = (
                    f"{priv}-mode write of 0x{written} to scounteren hit an Isla/Sailor type error "
                    f"before a clean legalize_scounteren path was produced."
                )
            else:
                summary = f"{priv}-mode write of 0x{written} to scounteren produced status {status}."
        elif exp == "counter_access_effect":
            if status == "success":
                path_name = f"{priv}-mode {counter} access succeeds"
                summary = (
                    f"{priv}-mode reads {counter}. The trace succeeds under a symbolic condition "
                    f"where the relevant counter-enable bits allow access."
                )
            elif status == "trapped":
                path_name = f"{priv}-mode {counter} access traps"
                summary = (
                    f"{priv}-mode reads {counter}. The trace traps under a symbolic condition "
                    f"where the relevant counter-enable bits do not allow access."
                )
            else:
                path_name = f"{priv}-mode {counter} access {status}"
                summary = f"{priv}-mode {counter} access produced status {status}."
        else:
            path_name = f"unknown scounteren path"
            summary = f"Trace produced status {status}."

        rows.append({
            "experiment": exp,
            "path_id": f"{path.stem}__path_{i:03d}",
            "trace_file": str(path),
            "path_name": path_name,
            "privilege": priv,
            "instruction": instr,
            "counter": counter,
            "written_value": written,
            "status": status,
            "exception": exception,
            "tool_error": terr,
            "summary": summary,
            "important_trace_events": important_events(block),
        })

with OUT_JSONL.open("w", encoding="utf-8") as f:
    for row in rows:
        f.write(json.dumps(row) + "\n")

lines = []
for idx, row in enumerate(rows, 1):
    lines.append(f"Path {idx}: {row['path_name']}")
    lines.append(f"Path ID: {row['path_id']}")
    lines.append(f"Experiment: {row['experiment']}")
    lines.append(f"Trace file: {row['trace_file']}")
    lines.append("")
    lines.append("Condition:")
    lines.append(f"- Privilege: {row['privilege']}.")
    lines.append(f"- Instruction: {row['instruction']}.")
    if row["counter"]:
        lines.append(f"- Counter: {row['counter']}.")
    if row["written_value"]:
        lines.append(f"- Written value: 0x{row['written_value']}.")
    lines.append("")
    lines.append("Outcome:")
    lines.append(f"- Status: {row['status']}.")
    lines.append(f"- Exception: {row['exception']}.")
    lines.append(f"- Tool error: {row['tool_error']}.")
    lines.append("")
    lines.append("Summary:")
    lines.append(row["summary"])
    lines.append("")
    lines.append("Important trace events:")
    for e in row["important_trace_events"]:
        lines.append(f"- {e}")
    lines.append("")
    lines.append("-" * 80)
    lines.append("")

OUT_TXT.write_text("\n".join(lines), encoding="utf-8")

print("total rows:", len(rows))
print("by experiment:", Counter(r["experiment"] for r in rows))
print("by status:", Counter(r["status"] for r in rows))
print("wrote:", OUT_JSONL)
print("wrote:", OUT_TXT)

total rows: 35
by experiment: Counter({'counter_access_effect': 24, 'write_legalize_probe': 8, 'csr_read_access': 3})
by status: Counter({'success': 14, 'trapped': 13, 'tool_error': 8})
wrote: scounteren_path_summaries.jsonl
wrote: scounteren_path_summaries.txt


In [2]:
from pathlib import Path
import re

INPUT_FILE = Path("rule.txt")
OUTPUT_FILE = Path("rule_clean.txt")

text = INPUT_FILE.read_text(errors="ignore")

rule_pattern = re.compile(
    r"(?m)^RULE(?:\s+Path)?\s+\d+\s*:\s*"
)

matches = list(rule_pattern.finditer(text))

if not matches:
    raise ValueError("No RULE lines found in rule.txt")

blocks = []

for i, m in enumerate(matches):
    start = m.start()
    end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
    block = text[start:end].strip()
    blocks.append(block)

seen = set()
clean_rules = []

for block in blocks:
    body = re.sub(r"^RULE(?:\s+Path)?\s+\d+\s*:\s*", "", block, flags=re.IGNORECASE).strip()

    key = re.sub(r"\s+", " ", body).strip().lower()

    if key in seen:
        continue

    seen.add(key)
    clean_rules.append(body)

output_lines = []

for i, body in enumerate(clean_rules, 1):
    output_lines.append(f"RULE {i}: {body}")

OUTPUT_FILE.write_text("\n".join(output_lines) + "\n")

print("input rules:", len(blocks))
print("unique rules:", len(clean_rules))
print("removed duplicates:", len(blocks) - len(clean_rules))
print("wrote:", OUTPUT_FILE)

input rules: 35
unique rules: 31
removed duplicates: 4
wrote: rule_clean.txt
